### scRNA-seq Tutorial 1: Data processing 

In this tutorial, you will learn the basic steps of processing scRNA-seq data which are performed prior to further analysis such clustering and annotation. 

We will be using sequencing data representing days 5 and 10 of differentiation towards lung progenitors following the protocol described in [PMID 3802619](https://www.sciencedirect.com/science/article/pii/S2589004223022824?via%3Dihub).

First, we import the packages we need to access the functions we use. If you need to use another package, install it and import it below. The main package we use to process single cell RNA-seq data is [scanpy](https://scanpy.readthedocs.io/en/stable/), which we install in the next cell.

In [ ]:
%pip install scanpy 

In [ ]:
import os
import sys
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

We will load the count matrix with the cells from day 5 of the differentiation into a variable called adata. 

In [ ]:
adata = sc.read_h5ad ('data/3802619/GSE167011_IPS_lung_differentiation_day5_raw.h5ad')

Single cell RNAseq count matrices are stored in an AnnData obect. It stores the cell-level information in .obs, and the gene-level information in .var. To see the gene names, we use ```adata.var``` to show the gene-level dataframe, where the gene names are used as indices. 

In [ ]:
adata.var

If we use ```adata.obs``` we will see the cell-level information, namely the cell barcodes.

In [ ]:
adata.obs

Lastly, using ```adata.X``` shows us the actual counts, in the form of an array of cells x genes. Based on this, <b> how many cells are in the dataset? </b>

In [ ]:
adata.X

First, let's add a cell-level annotation, namely the differentiation day, so we can later differentiate between the two samples. We are currently looking at day 5.

In [ ]:
adata.obs["day"] = 'day 5'
adata.obs

## Filtering

There are a few standard QC metrics we check in scRNAseq data at the cell level: 
1. ```n_genes_by_counts```: The number of genes which have >0 reads in the cell. 
2. ```total_counts```: The total number of reads in the cell.

If these numbers are too low, it could indicate empty droplets or poor quality cells. If the numbers are too big, it could point to posisble doublets or multiplets, meaning more than 1 cell was captured in the oil droplet. 

3. ```pct_counts_mit```: Percentage of counts coming from mitochondrial genes. A high mitochondrial fraction usually means stressed or dying cells. 
4. ```pct_counts_ribo```: Percentage of counts coming from ribosomal genes. Can help identify cells dominated by ribosomal transcripts. 

Let's first look at ```n_genes_by_counts```. We will calculate the number of genes with min 0 reads per cell using scanpy's in-built function and we will look at the distribution of this metric across our dataset.

In [ ]:
adata.obs["n_genes_by_counts"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()
adata.obs["n_genes_by_counts"].head()

In [ ]:
sc.pl.violin(
    adata,
    keys="n_genes_by_counts",
    jitter=0.4,
    show=False
)

We see a skewed distribution with a long tail at the high end. In scRNAseq, we have to manually inspect these metrics to understand what our data looks like and choose filtering threshold accordingly. In this case, considering we wish to filter out possible empty droplets and multiplets, we would set out thresholds in such a way as to remove most cells populating the distribution's tails. However, we must look at the measurements as a whole to make a reliable decision.  

We will now compute the remaining three metrics and plot them all. 

In [ ]:
# total_counts
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()

with open("data/mt_genes.txt") as f:
    mt_genes = [line.strip() for line in f]


# mitochondrial genes start with MT-
mito_mask = adata.var_names.isin(mt_genes)

# ribosomal genes start with RPS and RPL
ribo_mask = adata.var_names.str.startswith(("RPS", "RPL"))

# mitochondrial counts
mito_counts = np.asarray(adata.X[:, mito_mask].sum(axis=1)).ravel()

# ribosomal counts
ribo_counts = np.asarray(adata.X[:, ribo_mask].sum(axis=1)).ravel()

# percentages
adata.obs["pct_counts_mit"] = mito_counts / adata.obs["total_counts"] * 100
adata.obs["pct_counts_ribo"] = ribo_counts / adata.obs["total_counts"] * 100

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mit", "pct_counts_ribo"]

for i, metric in enumerate(metrics):
    sc.pl.violin(
        adata,
        keys=metric,
        jitter=0.4,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(metric)

plt.tight_layout()
plt.show()

In [ ]:
# Copy raw AnnData
adata_umap = adata.copy()
adata_umap.X = np.nan_to_num(adata_umap.X, nan=0.0)


# Compute UMAP from the cleaned raw counts
sc.pp.normalize_total(adata_umap, target_sum=1e4)
sc.pp.log1p(adata_umap)
sc.pp.pca(adata_umap)
sc.pp.neighbors(adata_umap)
sc.tl.umap(adata_umap)

# Make sure the QC metrics are present in the UMAP object
metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mit", "pct_counts_ribo"]
adata_umap.obs[metrics] = adata.obs.loc[adata_umap.obs_names, metrics]

# Plot 4 UMAPs side-by-side
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for i, metric in enumerate(metrics):
    sc.pl.umap(
        adata_umap,
        color=metric,
        ax=axes[i],
        show=False,
        title=metric
    )

plt.tight_layout()
plt.show()

Now we can see how our metrics are distributed among the dataset and look to see whether there are subsets of cells that we can filter out. In this case, we see one main cluster, from where two smaller clusters branch out which appear to have higher counts of expressed genes and total counts overall. However, the large cluster has ~500 genes per sample, which is significantly lower than expected. It is possible, in this case, that those cells are of low quality, and the real distribution can be seen once we remove them. 

Furthermore, there is a separate subset of cells which appears to have higher than average mitochondrial gene counts (40-60%), and a different subset with high ribosomal gene counts (~40). We apply our thresholds so as to filter these out. Can you tell how we got to these numbers?

In [ ]:
# example thresholds — change these after inspecting your violins/UMAPs
min_genes = 400 
max_genes = 2500 

#min_counts = 2000
#max_counts = 3500

max_pct_mit = 40
max_pct_ribo = 40

cells_keep = (
    (adata.obs["n_genes_by_counts"] >= min_genes) &
    (adata.obs["n_genes_by_counts"] <= max_genes) &
    #(adata.obs["total_counts"] >= min_counts) &
    #(adata.obs["total_counts"] <= max_counts) &
    (adata.obs["pct_counts_mit"] <= max_pct_mit) &
    (adata.obs["pct_counts_ribo"] <= max_pct_ribo)
)

adata = adata[cells_keep].copy()

print(adata.shape)


We can re-assess our distributions now, and look at how many cells are left.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mit", "pct_counts_ribo"]

for i, metric in enumerate(metrics):
    sc.pl.violin(
        adata,
        keys=metric,
        jitter=0.4,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(metric)

plt.tight_layout()
plt.show()

print (adata.X.shape)

## Normalization

Next, we have to normalize the read counts to make the cells comparable. Each cell in a scRNA-seq dataset has a different total number of reads. Without normalization, if two cells are biologically identical, but one was sequenced deeper and thus has overall more reads, it will look like it has higher expression in downstream analysis. Therefore, we must correct for that. 

We do this by scaling the data such that each cell's total count adds up to <b> 10 000 </b>, and then we can take the log to balance the variance.

In [ ]:
#Now use scanpy to normalize
sc.pp.normalize_total(adata, target_sum=1e4) #normalize so that each count adds up to 10,000 reads per cell
print (adata.X.sum(axis =1))

sc.pp.log1p(adata) #convert to log
print (adata.X.sum(axis =1))


For good practice, save your processed dataset so you do not need to redo the steps next time.

In [ ]:
adata.write_h5ad("data/3802619/GSE167011_IPS_lung_differentiation_day5_processed.h5ad")

## Your turn

Load the gene expression data for day 10 of the differentiation process and go through the filtering and normalization steps.

In [ ]:
#Load the data 
adata = sc.read_h5ad ('data/3802619/GSE167011_IPS_lung_differentiation_day10_raw.h5ad')

In [ ]:
#Filtering


In [ ]:
#Normalization


In [ ]:
#Save processed dataset
adata.write_h5ad("data/3802619/GSE167011_IPS_lung_differentiation_day10_processed.h5ad")

## Merge the different days

For further analysis, depending on your goal, you can work with the data separately or combine it. In this case, we will combine the days for reference mapping.

In [ ]:
# Load the two datasets if not in memory
adata = sc.read_h5ad ('data/3802619/GSE167011_IPS_lung_differentiation_day5_processed.h5ad')
adata_2 = sc.read_h5ad ('data/3802619/GSE167011_IPS_lung_differentiation_day10_processed.h5ad')


In [ ]:
adata_concat = sc.concat(
    [adata, adata_2],
    label="day",
    keys=["day5", "day10"]
)

adata_concat.obs

At later stages we will be filtering by highly variable genes and scaling the data. This is suitable for visualisation and clustering, but not for all analyses, therefore we will store our current gene expression matrix in a new attribute, ```adata.raw```.

In [ ]:
#Save
adata_concat.raw = adata_concat

## Highly variable genes

In scRNAseq data analysis, we mostly use the highly variable genes (HVGs) for downstream analysis, as they carry the biological differences between the cells. Therefore, we filter out the genes which are not variable to reduce noise. 

We use scanpy's in-built function to compute and visualise the gene variability. There is an inherent trend to gene expression data, where the higher the mean, the higher the variance of a gene. Scanpy then fits a curve to show the expected variance of each gene based on its mean. If the gene has higher variability then expected, we consider it a highly variable gene.

In [ ]:
#Compute mean and dispersion
sc.pp.highly_variable_genes(adata_concat, batch_key="day")
sc.pl.highly_variable_genes(adata_concat)

In [ ]:
#Subset to keep only genes that have been marked as highly variable
adata_concat = adata_concat[:, adata_concat.var.highly_variable].copy()
print (adata_concat.X.shape)

## Visualisation via UMAP

For visualizing single cell expression data, we use UMAPs. Each dot is a cell, and the 2d representation highlights local clusters and maintains trajectories. Below, visualise your data labeled by day. 

Note that PCA on the highly variable genes is commonly used to preprocess data before applying UMAP to reduce noise and computational load, allowing UMAP to focus on preserving non-linear, local structures. For this reason, we first scale and run PCA on our data. The scaled values are useful for visualisation and clustering, but should not be used to examine gene expression signatures. Furthermore, since we filtered by highly variable genes, we do not have the expression of all genes in our adat object anymore.  We will therefore use the stored log-normalized values to show the gene expression by specifying ```use_raw = True``` when plotting.

In [ ]:
#Plot the clusters
sc.pp.scale(adata_concat)
sc.pp.pca(adata_concat)
sc.pp.neighbors(adata_concat)
sc.tl.umap(adata_concat)
sc.pl.umap(
    adata_concat,
    color=["day"],
    legend_loc="best",
)

We can also use the UMAP to visualise the gene expression distribution in our dataset. 

<b>Go to the paper and complete the gene list below with genes you expect to be expressed at the two differentiation stages (day 5 and day 10). </b>

In [ ]:
gene_list = ["CER1", "CXCR4", "FOXA2", "APOA1", "AFP"]

sc.pl.umap(
    adata_concat,
    color=gene_list,
    legend_loc="best",
    use_raw = True
)

In [103]:
#Save the fully processed adata so it is easy to load and use later
adata_concat.write_h5ad("data/3802619/GSE167011_IPS_lung_differentiation_concat.h5ad")